# Spike llama.cpp sm_75 (Task 0.2): build, bench, đóng gói

Chạy **một lần**, từ trên xuống dưới (Run All), trên Kaggle **GPU T4 x2** với **Internet bật**. Notebook chỉ chạy rồi kết thúc; xong thì **Stop session** (Kaggle ToS: không giữ notebook chạy).

Kết quả:

1. `/kaggle/working/llamacpp-sm75-bin/` : binary `llama-server`, `llama-bench` cùng mọi `.so` của llama.cpp/ggml, `MANIFEST.txt`, `dataset-metadata.json`. Dùng làm Dataset private `llamacpp-sm75-bin` (đúng thứ `kaggle/serve.ipynb` cell 2 tìm).
2. `/kaggle/working/qwen3-8b-gguf/` : `Qwen3-8B-Q5_K_M.gguf` + `dataset-metadata.json`. Dùng làm Dataset private `qwen3-8b-gguf`.
3. Cell cuối in khối `RESULT`: copy nguyên khối đó gửi lại cho trợ lý.

Thư viện CUDA runtime (`libcudart`, `libcublas`...) **không** đóng gói: chúng nằm sẵn trong image Kaggle. Đặt `BUNDLE_CUDA_LIBS = True` ở cell 1 nếu muốn đóng gói luôn (nặng, hàng trăm MB).

In [ ]:
# [1] config + pure helpers (no side effects; unit-tested in kaggle/tests)
import hashlib, json, os, re, shutil, subprocess, time
from pathlib import Path

# llama.cpp release tag, verified to exist on github.com/ggml-org/llama.cpp/releases on 2026-11-23
# (commit f5e6915c28f10330885fcb9243a93ae98d3115ab). Set to "master" only for a quick try: that is UNPINNED.
LLAMA_CPP_REF = "b12651"
LLAMA_CPP_COMMIT = "f5e6915c28f10330885fcb9243a93ae98d3115ab"  # soft check, ignored when ref is "master"
# LLAMA_CURL is deprecated in current llama.cpp (warning only), so it is not passed. OpenSSL off: models come via huggingface_hub.
CMAKE_FLAGS = "-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_OPENSSL=OFF -DCMAKE_BUILD_TYPE=Release"
PASS_TG_TPS = 15  # pass rule: token-generation tok/s of the Q5_K_M model on GPU 0
BUNDLE_CUDA_LIBS = False

PKG = Path("/kaggle/working/llamacpp-sm75-bin")
GGUF_DIR = Path("/kaggle/working/qwen3-8b-gguf")
GGUF_Q4_DIR = Path("/kaggle/working/qwen3-8b-gguf-q4-fallback")  # NOT part of the dataset
TMP = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
SRC = TMP / "llama.cpp"
GGUF_REPOS = ["Qwen/Qwen3-8B-GGUF", "unsloth/Qwen3-8B-GGUF"]  # official first, unsloth only if the official file is missing
Q5, Q4 = "Qwen3-8B-Q5_K_M.gguf", "Qwen3-8B-Q4_K_M.gguf"
FLAGS_CHECKED = ["--jinja", "-fa", "--flash-attn", "--chat-template-kwargs", "--api-key",
                 "--reasoning-budget", "-np", "-c", "--host", "--port"]


def parse_bench(text):
    # llama-bench -o json -> (pp tok/s, tg tok/s); None when a row is missing (crash, OOM)
    m = re.search(r"\[\s*\{", text)
    if not m or "]" not in text:
        return None, None
    rows = json.loads(text[m.start(): text.rindex("]") + 1])
    pp = tg = None
    for r in rows:
        if r["n_prompt"] > 0 and r["n_gen"] == 0:
            pp = r["avg_ts"]
        elif r["n_gen"] > 0 and r["n_prompt"] == 0:
            tg = r["avg_ts"]
    return pp, tg


def parse_ldd(text):
    found, missing = {}, []
    for line in text.splitlines():
        m = re.match(r"\s*(\S+) => (\S+) \(0x", line)
        if m:
            found[m.group(1)] = m.group(2)
        elif "=> not found" in line:
            missing.append(line.split()[0])
    return found, missing


def check_flags(help_text, flags=FLAGS_CHECKED):
    return {f: bool(re.search(r"(?<![\w-])" + re.escape(f) + r"(?![\w-])", help_text)) for f in flags}


def help_line(help_text, flag):
    return next((ln.strip() for ln in help_text.splitlines() if re.search(r"(?<![\w-])" + re.escape(flag) + r"(?![\w-])", ln)), "")

## 2. Môi trường (GPU, CUDA, cmake)

In [ ]:
# [2] environment info
os.environ["PATH"] += ":/usr/local/cuda/bin"
R = {}  # everything the final RESULT block prints


def sh(cmd, env=None, cwd=None, check=True, echo=True, merge=True, timeout=None):
    p = subprocess.run(cmd, env=env, cwd=cwd, text=True, errors="replace", timeout=timeout,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT if merge else subprocess.PIPE)
    if echo and p.stdout:
        print(p.stdout.rstrip())
    if check and p.returncode != 0:
        tail = "\n".join((p.stdout or "").splitlines()[-60:])
        raise RuntimeError(f"{cmd[0]} failed (exit {p.returncode}). Last 60 lines:\n{tail}")
    return p


sh(["nvidia-smi"])
gpus = sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader"], echo=False).stdout.strip().splitlines()
assert gpus, "no GPU: Settings > Accelerator must be GPU T4 x2"
R["gpus"] = [g.split(",")[0].strip() for g in gpus]
R["gpu_detail"] = gpus
nvcc = sh(["nvcc", "--version"]).stdout
R["cuda"] = (re.search(r"release (\d+\.\d+)", nvcc) or [None, "unknown"])[1]
R["cmake"] = sh(["cmake", "--version"]).stdout.splitlines()[0]
R["nproc"] = int(sh(["nproc"], echo=False).stdout.strip())
print("GPUs:", R["gpus"], "| CUDA toolkit", R["cuda"], "|", R["cmake"], "| nproc", R["nproc"])

## 3. Clone llama.cpp (ref ghim cố định)

In [ ]:
# [3] clone at the pinned ref
if LLAMA_CPP_REF == "master":
    print("WARNING: LLAMA_CPP_REF = master is UNPINNED. Pin it to the tag/commit in the RESULT block after this build works.")
if SRC.exists():
    shutil.rmtree(SRC)
sh(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_REF, "https://github.com/ggml-org/llama.cpp", str(SRC)])
R["ref"] = LLAMA_CPP_REF
R["commit"] = sh(["git", "-C", str(SRC), "rev-parse", "HEAD"], echo=False).stdout.strip()
if LLAMA_CPP_REF != "master" and R["commit"] != LLAMA_CPP_COMMIT:
    print("WARNING: commit differs from LLAMA_CPP_COMMIT (tag moved?):", R["commit"])
print("llama.cpp", R["ref"], R["commit"])

## 4. Build (CUDA, chỉ sm_75)

Mất khoảng 20-30 phút, cell không in gì giữa chừng. Cell tìm stub `libcuda.so` của CUDA toolkit **trước khi configure** (CUDA 12: `lib64/stubs`, CUDA 13: `targets/x86_64-linux/lib/stubs`), đưa vào `-DCMAKE_LIBRARY_PATH` để CMake tạo được `CUDA::cuda_driver`; nếu configure vẫn báo `cuda_driver` thì thử lại với biến cache chỉ thẳng tới file stub. Stub chỉ dùng lúc build, không bao giờ đưa vào `LD_LIBRARY_PATH` lúc chạy hay vào gói.

In [ ]:
# [4] find the libcuda stub, configure (with retry variants), build
import glob


def build_log_fail(what, p):
    Path("/kaggle/working/build.log").write_text(p.stdout)
    print("\n".join(p.stdout.splitlines()[-60:]))
    print("cmake:", R["cmake"], "| libcuda stubs found:", found or "none")
    raise RuntimeError(f"{what} failed: paste the lines above (full log: /kaggle/working/build.log)")


# CUDA 12: /usr/local/cuda/lib64/stubs ; CUDA 13: /usr/local/cuda/targets/x86_64-linux/lib/stubs
found = sorted({os.path.realpath(x) for x in glob.glob("/usr/local/cuda*/**/stubs/libcuda.so*", recursive=True)})
print("libcuda stubs found:", found or "none")
if not found:
    for d in sorted(glob.glob("/usr/local/cuda*") + glob.glob("/usr/local/cuda*/targets/*/lib") + glob.glob("/usr/local/cuda*/lib64")):
        print(d, "->", sorted(os.listdir(d))[:40])
stub_file = next((x for x in found if x.endswith("libcuda.so")), found[0] if found else None)
stub_dir = TMP / "cuda-stub"  # private dir, build-only: the real driver must win at runtime
shutil.rmtree(stub_dir, ignore_errors=True)
stub_env = {}
variants = [("cmake-library-path", [])]
if stub_file:
    stub_dir.mkdir()
    for n in ("libcuda.so", "libcuda.so.1"):
        os.symlink(stub_file, stub_dir / n)
    stub_env = {"LIBRARY_PATH": str(stub_dir), "LD_LIBRARY_PATH": f"{stub_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}".rstrip(":")}
    variants = [("cmake-library-path", [f"-DCMAKE_LIBRARY_PATH={stub_dir}"]),
                # variable names differ between CMake/FindCUDAToolkit versions, so set both spellings
                ("cuda-driver-library", [f"-DCMAKE_LIBRARY_PATH={stub_dir}", "-DCUDAToolkit_ROOT=/usr/local/cuda",
                                         f"-DCUDA_cuda_driver_LIBRARY={stub_file}", f"-DCUDA_DRIVER_LIBRARY={stub_file}"])]
R["stub_dir"], R["stub_file"], R["configure_variant"] = str(stub_dir) if stub_file else None, stub_file, None
bld_env = dict(os.environ, **stub_env)

t0 = time.time()
for name, extra in variants:
    shutil.rmtree(SRC / "build", ignore_errors=True)
    p = sh(["cmake", "-B", "build", *CMAKE_FLAGS.split(), *extra], cwd=SRC, env=bld_env, check=False, echo=False)
    if p.returncode == 0:
        R["configure_variant"] = name
        print("configure OK, variant:", name)
        break
    print(f"configure failed with variant {name}")
    if "cuda_driver" not in p.stdout:
        break  # a different error: retrying with other libcuda variables will not help
if R["configure_variant"] is None:
    build_log_fail("cmake configure", p)
p = sh(["cmake", "--build", "build", "--target", "llama-server", "llama-bench", "-j", str(R["nproc"])],
       cwd=SRC, env=bld_env, check=False, echo=False)
R["build_min"] = round((time.time() - t0) / 60, 1)
if p.returncode != 0:
    build_log_fail("build", p)
BIN = SRC / "build" / "bin"
assert (BIN / "llama-server").is_file() and (BIN / "llama-bench").is_file(), "binaries missing in build/bin"
print(f"build OK in {R['build_min']} min (stub: {R['stub_file']}, configure variant: {R['configure_variant']})")
print(sorted(f.name for f in BIN.iterdir()))

## 5. Đóng gói `llamacpp-sm75-bin`

Chép `build/bin/*` (đã bỏ symlink, mỗi `.so` thành file thật dưới đúng tên mà loader tìm) vào `/kaggle/working/llamacpp-sm75-bin/`, rồi chạy thử **từ thư mục đóng gói** với `LD_LIBRARY_PATH` trỏ vào đó.

In [ ]:
# [5] package build/bin/* + every needed .so, then verify it runs from the package dir
if PKG.exists():
    shutil.rmtree(PKG)
PKG.mkdir(parents=True)
# prepend like serve.ipynb does: keep Kaggle's own LD_LIBRARY_PATH (nvidia/cuda dirs)
build_env = dict(os.environ, LD_LIBRARY_PATH=f"{BIN}:{os.environ.get('LD_LIBRARY_PATH', '')}".rstrip(":"))
needed = {}
for exe in ("llama-server", "llama-bench"):
    found, missing = parse_ldd(sh(["ldd", str(BIN / exe)], env=build_env, echo=False).stdout)
    needed.update(found)
    print(exe, "needs", len(found), "libs; unresolved at build dir:", missing)
real_bin = os.path.realpath(BIN)
cuda_rt = {n: p for n, p in needed.items() if os.path.realpath(p).startswith("/usr/local/cuda")}
for f in sorted(BIN.iterdir()):
    if f.is_file() and (not f.is_symlink() or f.name in needed):  # symlink only if a binary asks for that exact name
        shutil.copy2(f, PKG / f.name)  # follows symlinks -> real file
if BUNDLE_CUDA_LIBS:
    for n, p in cuda_rt.items():
        shutil.copy2(p, PKG / n)
for f in PKG.iterdir():
    if not f.name.endswith(".so") and ".so." not in f.name:
        f.chmod(0o755)

run_env = dict(os.environ, LD_LIBRARY_PATH=f"{PKG}:{os.environ.get('LD_LIBRARY_PATH', '')}".rstrip(":"), CUDA_VISIBLE_DEVICES="0")
in_tree = {n for n, p in needed.items() if os.path.realpath(p).startswith(real_bin)}
real_pkg = os.path.realpath(PKG)
missing_pkg, leaked = [], []
for exe in ("llama-server", "llama-bench"):
    found, m = parse_ldd(sh(["ldd", str(PKG / exe)], env=run_env, echo=False).stdout)
    missing_pkg += m
    # a llama/ggml lib resolved outside the package dir (e.g. via a baked-in build RPATH) would hide a missing file
    leaked += [n for n in in_tree if n in found and not os.path.realpath(found[n]).startswith(real_pkg)]
    leaked += [n for n, x in found.items() if "stubs" in x or (R["stub_dir"] and x.startswith(R["stub_dir"]))]  # libcuda stub must never win at runtime
assert not missing_pkg, f"shared libs not found when running from the package dir: {missing_pkg}"
assert not leaked, f"libs resolved outside the package dir: {leaked}"
version = sh([str(PKG / "llama-server"), "--version"], env=run_env, timeout=120).stdout.strip()
R["version"] = " | ".join(ln.strip() for ln in version.splitlines() if "version" in ln.lower() or "built" in ln.lower()) or version[:200]

lines = ["llama.cpp ref: " + R["ref"], "commit: " + R["commit"], "cmake flags: " + CMAKE_FLAGS,
         "llama-server --version:", version, "", "files (sha256  bytes  name):"]
for f in sorted(PKG.iterdir()):
    lines.append(f"{hashlib.sha256(f.read_bytes()).hexdigest()}  {f.stat().st_size}  {f.name}")
lines += ["", "CUDA runtime libs used (" + ("bundled" if BUNDLE_CUDA_LIBS else "NOT bundled, provided by the Kaggle image") + "):"]
lines += [f"{n} -> {p}" for n, p in sorted(cuda_rt.items())]
(PKG / "MANIFEST.txt").write_text("\n".join(lines) + "\n")
# Placeholder on purpose: replace <KAGGLE_USERNAME> with your Kaggle username only if you upload with the kaggle CLI.
# The Kaggle UI route (Output > New Dataset) ignores this file.
(PKG / "dataset-metadata.json").write_text(json.dumps(
    {"title": "llamacpp-sm75-bin", "id": "<KAGGLE_USERNAME>/llamacpp-sm75-bin", "licenses": [{"name": "other"}]}, indent=2) + "\n")
R["pkg_mb"] = round(sum(f.stat().st_size for f in PKG.iterdir()) / 1e6)
print("packaged:", sorted(f.name for f in PKG.iterdir()))
print("package size:", R["pkg_mb"], "MB | version:", R["version"])

## 6. Tải GGUF `Qwen3-8B-Q5_K_M`

Repo chính thức `Qwen/Qwen3-8B-GGUF` (đã kiểm tra có file này). Chỉ khi thiếu file mới dùng `unsloth/Qwen3-8B-GGUF`. Repo công khai nên không cần secret; nếu có secret `HF_TOKEN` thì dùng (không in ra).

In [ ]:
# [6] download Qwen3-8B Q5_K_M into the dataset folder
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    sh(["pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import hf_hub_download

hf_auth = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_auth = UserSecretsClient().get_secret("HF_TOKEN") or None
except Exception:
    pass  # secret not attached: the repos are public


def fetch(filename, dest):
    dest.mkdir(parents=True, exist_ok=True)
    for repo in GGUF_REPOS:
        try:
            path = hf_hub_download(repo, filename, local_dir=str(dest), token=hf_auth)
            shutil.rmtree(dest / ".cache", ignore_errors=True)  # keep the dataset folder clean
            return repo, Path(path)
        except Exception as e:
            print(f"{repo}/{filename}: {type(e).__name__}, trying next repo")
    raise RuntimeError(f"{filename} not found in {GGUF_REPOS}")


R["gguf_repo"], GGUF = fetch(Q5, GGUF_DIR)
R["gguf_mb"] = round(GGUF.stat().st_size / 1e6)
(GGUF_DIR / "dataset-metadata.json").write_text(json.dumps(
    {"title": "qwen3-8b-gguf", "id": "<KAGGLE_USERNAME>/qwen3-8b-gguf", "licenses": [{"name": "other"}]}, indent=2) + "\n")
print("GGUF:", GGUF, R["gguf_mb"], "MB from", R["gguf_repo"])

## 7. `llama-bench` (GPU 0)

`-ngl 99 -p 4096 -n 256`. Đạt khi **tg >= `PASS_TG_TPS` (15 tok/s)**. Không đạt (hoặc lỗi/OOM) thì tự bench thêm `Q4_K_M`.

In [ ]:
# [7] llama-bench on a single GPU, pass rule, automatic Q4_K_M fallback
def bench(gguf):
    p = sh([str(PKG / "llama-bench"), "-m", str(gguf), "-ngl", "99", "-p", "4096", "-n", "256", "-r", "3", "-o", "json"],
           env=run_env, check=False, echo=False, merge=False)
    pp, tg = parse_bench(p.stdout)
    if tg is None:
        print("bench failed (exit", p.returncode, "). stderr tail:\n" + "\n".join(p.stderr.splitlines()[-30:]))
    return pp, tg


R["bench"] = {"Q5_K_M": bench(GGUF)}
if not (R["bench"]["Q5_K_M"][1] or 0) >= PASS_TG_TPS:
    print(f"Q5_K_M tg below {PASS_TG_TPS} tok/s (or failed): also benchmarking Q4_K_M")
    _, GGUF_Q4 = fetch(Q4, GGUF_Q4_DIR)
    R["bench"]["Q4_K_M"] = bench(GGUF_Q4)


def fmt(v):
    return "n/a" if v is None else f"{v:.1f}"


R["pass"] = {q: (tg or 0) >= PASS_TG_TPS for q, (pp, tg) in R["bench"].items()}
print(f"{'quant':8} {'pp4096 tok/s':>13} {'tg256 tok/s':>12}  pass (tg >= {PASS_TG_TPS})")
for q, (pp, tg) in R["bench"].items():
    print(f"{q:8} {fmt(pp):>13} {fmt(tg):>12}  {'PASS' if R['pass'][q] else 'FAIL'}")

## 8. Cờ của `llama-server`

Kiểm tra các cờ mà `kaggle/serve.ipynb` và plan đang giả định có trong `--help` của bản vừa build (giải quyết rủi ro "cờ llama-server chưa xác minh").

In [ ]:
# [8] which llama-server flags exist in this build
p = sh([str(PKG / "llama-server"), "--help"], env=run_env, check=False, echo=False, timeout=120)
help_text = p.stdout
assert help_text.strip(), "llama-server --help printed nothing"
R["flags"] = check_flags(help_text)
R["flag_lines"] = {f: help_line(help_text, f) for f in FLAGS_CHECKED}
print(f"{'flag':24} {'present':8} help line")
for f, ok in R["flags"].items():
    print(f"{f:24} {'YES' if ok else 'NO':8} {R['flag_lines'][f][:90]}")

## 9. Khối RESULT

Copy toàn bộ khối bên dưới (cả hàng rào ```) gửi lại cho trợ lý. Sau đó: Save Version, tạo hai Dataset (xem `docs/spikes.md#llama`), **Stop session**.

In [ ]:
# [9] single copy-pasteable RESULT block
pinned = "pinned" if LLAMA_CPP_REF != "master" else "UNPINNED (master): pin to the commit below"
rows = [f"| {q} | {fmt(pp)} | {fmt(tg)} | {'PASS' if R['pass'][q] else 'FAIL'} |" for q, (pp, tg) in R["bench"].items()]
flag_rows = [f"| `{f}` | {'có' if ok else 'KHÔNG'} | `{R['flag_lines'][f][:80]}` |" for f, ok in R["flags"].items()]
out = [
    "```",
    "## RESULT Task 0.2 (llama.cpp sm_75)",
    f"- date (UTC): {time.strftime('%Y-%m-%d %H:%M', time.gmtime())}",
    f"- llama.cpp: ref `{R['ref']}` ({pinned}), commit `{R['commit']}`",
    f"- llama-server --version: {R['version']}",
    f"- CUDA toolkit: {R['cuda']} | {R['cmake']}",
    f"- GPUs: {'; '.join(R['gpu_detail'])}",
    f"- build: {R['build_min']} min with nproc={R['nproc']}, stub_dir `{R['stub_dir']}`, stub_file `{R['stub_file']}`, configure_variant `{R['configure_variant']}`",
    f"- packaged `llamacpp-sm75-bin`: {R['pkg_mb']} MB",
    f"- GGUF: `{Q5}` from `{R['gguf_repo']}`, {R['gguf_mb']} MB",
    "",
    f"| quant | pp4096 tok/s | tg256 tok/s | pass (tg >= {PASS_TG_TPS}) |",
    "|---|---|---|---|",
    *rows,
    "",
    "| flag | có/không | dòng help |",
    "|---|---|---|",
    *flag_rows,
    "```",
]
print("\n".join(out))